In [1]:
%load_ext autoreload
%autoreload 2

# Phase 1

In [ ]:
import os
import sys
import shutil
import gymnasium as gym
import torch

# Move up 2 levels: Notebooks/training_transformer/ -> Notebooks/ -> Project Root
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "../.."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print(f"Project root set to: {PROJECT_ROOT}")

from src.rl_transformer.env_adapter import MatchEnv
from src.rl_transformer.pool import PoolOpponentController
from src.rl_transformer.ppo import train_mappo
from src.rl_transformer.transformer_model import TransformerActorCritic

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 1v1 Geometry & Physical Parameters
TEAM_SIZE = 1
PITCH_W = 840.0
PITCH_H = 500.0
GOAL_H = 450.0
ROUND_STEPS = 900
ACTION_REPEAT = 6
NUM_ENVS = 24

SAVE_DIR_S1_P1 = "models/stage1/phase1"
POOL_DIR_S1_P1 = os.path.join(SAVE_DIR_S1_P1, "pool")
os.makedirs(POOL_DIR_S1_P1, exist_ok=True)

In [ ]:
def make_s1_p1_env(env_rank: int):
  def _thunk():
    opp_ctrl = PoolOpponentController(
        pool_dir=POOL_DIR_S1_P1,
        team="blue",
        device="cpu",
        p_random=0.3,    
        p_heuristic=0.0,  
    )
    env = MatchEnv(
        team_size=TEAM_SIZE,
        learner_team_size=TEAM_SIZE,
        opp_team_size=TEAM_SIZE,
        learner_team="red",
        max_round_steps=ROUND_STEPS,
        action_repeat=ACTION_REPEAT,
        goal_height=GOAL_H,
        pitch_width=PITCH_W,
        pitch_height=PITCH_H,
        opponent_controller=opp_ctrl,
        random_reset_opponents=["random"]
    )
    env.reset(seed=1000 + env_rank)
    return env
  return _thunk

envs_s1_p1 = gym.vector.AsyncVectorEnv(
    [make_s1_p1_env(i) for i in range(NUM_ENVS)],
    context="fork",
)

model_s1 = TransformerActorCritic().to(device)

# Seed initial self-play pool with randomly initialized weights
init_weights_path = os.path.join(POOL_DIR_S1_P1, "champion.pt")
torch.save(model_s1.state_dict(), init_weights_path)
torch.save(model_s1.state_dict(), os.path.join(POOL_DIR_S1_P1, "history_0.pt"))

print("🚀 Starting Stage 1 - Phase 1: Bootstrapping motor skills...")

train_mappo(
    envs=envs_s1_p1,
    model=model_s1,
    device=device,
    team_size=TEAM_SIZE,
    opp_team_size=TEAM_SIZE,
    total_timesteps=3_000_000,
    num_envs=NUM_ENVS,
    num_steps=256,
    update_epochs=3,
    minibatch_size=1024,
    lr_init=1e-3,
    lr_final=1e-5,
    ent_coef_init=0.020,
    ent_coef_final=0.001,
    gamma=0.99,
    gae_lambda=0.95,
    active_tiers=["random", "champion"],
    target_tier="champion",
    filter_thresholds={"random": 0.85},  # Must defeat random bots >50% to qualify
    tier_ratios={"random": 0.50, "champion": 0.50},
    eval_episodes=100,
    eval_freq=200_000,
    save_dir=SAVE_DIR_S1_P1,
    pool_dir=POOL_DIR_S1_P1,
    goal_height=GOAL_H,
    pitch_width=PITCH_W,
    pitch_height=PITCH_H,
    max_steps=ROUND_STEPS,
    action_repeat=ACTION_REPEAT,
)

envs_s1_p1.close()

# Phase 2

In [ ]:
SAVE_DIR_S1_P2 = "models/stage1/phase2"
POOL_DIR_S1_P2 = os.path.join(SAVE_DIR_S1_P2, "pool")

PITCH_W_REG = 1200.0
PITCH_H_REG = 800.0
GOAL_H_REG = 220.0  # Regulation net
ROUND_STEPS = 1800
ACTION_REPEAT = 6
os.makedirs(POOL_DIR_S1_P2, exist_ok=True)

phase1_best = os.path.join(SAVE_DIR_S1_P1, "best_model.pt")
phase1_final = os.path.join(SAVE_DIR_S1_P1, "final_model.pt")
seed_file = phase1_best if os.path.exists(phase1_best) else phase1_final

shutil.copy(seed_file, os.path.join(POOL_DIR_S1_P2, "champion.pt"))
shutil.copy(seed_file, os.path.join(POOL_DIR_S1_P2, "history_0.pt"))

print(f"🔥 Phase 2 seeded with weights from: {seed_file}")

In [ ]:
def make_s1_p2_env(env_rank: int):
  def _thunk():
    opp_ctrl = PoolOpponentController(
        pool_dir=POOL_DIR_S1_P2,
        team="blue",
        device="cpu",
        p_random=0.05,     
        p_heuristic=0.65,  
    )
    env = MatchEnv(
        team_size=TEAM_SIZE,
        learner_team_size=TEAM_SIZE,
        opp_team_size=TEAM_SIZE,
        learner_team="red",
        max_round_steps=ROUND_STEPS,
        action_repeat=ACTION_REPEAT,
        goal_height=GOAL_H_REG,
        pitch_width=PITCH_W_REG,
        pitch_height=PITCH_H_REG,
        opponent_controller=opp_ctrl,
        opponent_stats=[
            (3200.0, 1200.0),
        ],
    )
    env.reset(seed=2000 + env_rank)
    return env
  return _thunk

envs_s1_p2 = gym.vector.AsyncVectorEnv(
    [make_s1_p2_env(i) for i in range(NUM_ENVS)],
    context="fork",
)

model_s1_p2 = TransformerActorCritic().to(device)
ckpt = torch.load(seed_file, map_location=device, weights_only=False)
state_dict = ckpt["model_state_dict"] if isinstance(ckpt, dict) and "model_state_dict" in ckpt else ckpt
model_s1_p2.load_state_dict(state_dict, strict=True)
print("✅ Phase 2 model weights loaded successfully.")

train_mappo(
    envs=envs_s1_p2,
    model=model_s1_p2,
    device=device,
    team_size=TEAM_SIZE,
    opp_team_size=TEAM_SIZE,
    total_timesteps=5_000_000,
    num_envs=NUM_ENVS,
    num_steps=256,
    update_epochs=3,
    minibatch_size=1024,
    lr_init=3e-5,
    lr_final=3e-6,
    ent_coef_init=0.008,
    ent_coef_final=0.002,
    gamma=0.995,
    gae_lambda=0.96,
    active_tiers=["heuristic"],
    target_tier="heuristic",
    #filter_thresholds={"heuristic": 0.90},  
    #tier_ratios={"heuristic": 0.50, "champion": 0.50},
    eval_episodes=100,
    eval_freq=200_000,
    save_dir=SAVE_DIR_S1_P2,
    pool_dir=POOL_DIR_S1_P2,
    goal_height=GOAL_H_REG,
    pitch_width=PITCH_W_REG,
    pitch_height=PITCH_H_REG,
    max_steps=ROUND_STEPS,
    action_repeat=ACTION_REPEAT,
)

envs_s1_p2.close()

# Phase 3

In [5]:
import os
import shutil
import sys
import gymnasium as gym
import torch

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "../.."))
if PROJECT_ROOT not in sys.path:
  sys.path.insert(0, PROJECT_ROOT)

from src.rl_transformer.env_adapter import MatchEnv
from src.rl_transformer.pool import PoolOpponentController
from src.rl_transformer.ppo import train_mappo
from src.rl_transformer.transformer_model import TransformerActorCritic

# ── Hardware Performance Flags ──
torch.backends.cudnn.benchmark = True
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ── Regulation Dimensions & Physics ──
TEAM_SIZE = 1
PITCH_W_REG = 1200.0
PITCH_H_REG = 800.0
GOAL_H_REG = 220.0
ROUND_STEPS = 2400
ACTION_REPEAT = 6
NUM_ENVS = 24

# ── Directories & Seed Checkpoints ──
SAVE_DIR_S1_P2 = "models/stage1/phase2"
SAVE_DIR_S1_P3 = "models/stage1/phase3"
POOL_DIR_S1_P3 = os.path.join(SAVE_DIR_S1_P3, "pool")
os.makedirs(POOL_DIR_S1_P3, exist_ok=True)

phase2_best = os.path.join(SAVE_DIR_S1_P2, "best_model.pt")
phase2_final = os.path.join(SAVE_DIR_S1_P2, "final_model.pt")
seed_file = phase2_best if os.path.exists(phase2_best) else phase2_final

if not os.path.exists(seed_file):
  raise FileNotFoundError(f"Missing Phase 2 checkpoint: {seed_file}")

shutil.copy(seed_file, os.path.join(POOL_DIR_S1_P3, "champion.pt"))
shutil.copy(seed_file, os.path.join(POOL_DIR_S1_P3, "history_0.pt"))
print(f"🔥 Phase 3 seeded from Phase 2 champion: {seed_file}")


# ── Environment Factory ──
def make_s1_p3_env(env_rank: int):
  def _thunk():
    torch.set_num_threads(1)

    opp_ctrl = PoolOpponentController(
        pool_dir=POOL_DIR_S1_P3,
        team="blue",
        device="cpu",
        p_random=0.05,
        # 45% Heuristic reduces CPU model inference load while keeping motor reflexes sharp
        p_heuristic=0.45,
        frame_stack=3,
    )
    env = MatchEnv(
        team_size=TEAM_SIZE,
        learner_team_size=TEAM_SIZE,
        opp_team_size=TEAM_SIZE,
        learner_team="red",
        max_round_steps=ROUND_STEPS,
        action_repeat=ACTION_REPEAT,
        goal_height=GOAL_H_REG,
        pitch_width=PITCH_W_REG,
        pitch_height=PITCH_H_REG,
        opponent_controller=opp_ctrl,
        opponent_stats=[
            (3200.0, 1200.0),
        ],
        frame_stack=3,
    )
    env.reset(seed=3000 + env_rank)
    return env

  return _thunk


# Direct OS pipes (shared_memory=False) avoid semaphore lock contention on small dicts
envs_s1_p3 = gym.vector.AsyncVectorEnv(
    [make_s1_p3_env(i) for i in range(NUM_ENVS)],
    context="fork",
    shared_memory=False,
)

# ── Model Initialization & Warmstart ──
model_s1_p3 = TransformerActorCritic().to(device)
ckpt = torch.load(seed_file, map_location=device, weights_only=False)
state_dict = (
    ckpt["model_state_dict"]
    if isinstance(ckpt, dict) and "model_state_dict" in ckpt
    else ckpt
)
model_s1_p3.load_state_dict(state_dict, strict=True)
print("✅ Phase 3 model weights warmstarted successfully.")

# ── Accelerated Self-Play PPO Training ──
train_mappo(
    envs=envs_s1_p3,
    model=model_s1_p3,
    device=device,
    team_size=TEAM_SIZE,
    opp_team_size=TEAM_SIZE,
    total_timesteps=30_000_000,
    num_envs=NUM_ENVS,
    num_steps=512,          # 12 * 512 = 6,144 batch size
    update_epochs=3,
    minibatch_size=2048,    # 3 minibatches per epoch (fast GPU saturation)
    lr_init=3e-5,
    lr_final=2e-6,
    ent_coef_init=0.015,
    ent_coef_final=0.001,
    gamma=0.996,
    gae_lambda=0.97,
    active_tiers=["heuristic", "champion"],
    target_tier="champion",
    filter_thresholds={"heuristic": 0.85},
    tier_ratios={"heuristic": 0.50, "champion": 0.50},
    eval_episodes=100,
    eval_freq=250_000,
    save_dir=SAVE_DIR_S1_P3,
    pool_dir=POOL_DIR_S1_P3,
    goal_height=GOAL_H_REG,
    pitch_width=PITCH_W_REG,
    pitch_height=PITCH_H_REG,
    max_steps=ROUND_STEPS,
    action_repeat=ACTION_REPEAT,
)

envs_s1_p3.close()

Using device: cuda
🔥 Phase 3 seeded from Phase 2 champion: models/stage1/phase2/best_model.pt
✅ Phase 3 model weights warmstarted successfully.
🚀 Entity-Transformer MAPPO Initialized | Format: 1v1 | Envs: 24 | Batch: 12288 | Device: cuda

📊 [EVALUATION @ Step 258,048 | Rollout SPS: 2304 | Tiers: ['heuristic', 'champion']]
   ⚔️  vs Heuristic [FILTER] | WR:  96.0% | Reward: +3.866 | Goals: 141 Scored, 8 Conceded (+133 Net)
   ⚔️  vs Champion  [TARGET] | WR:  36.0% | Reward: -0.046 | Goals: 31 Scored, 29 Conceded (+2 Net)
   ❌ Retaining current baseline. Did not pass criteria for champion: [WR: 36.0%, Reward: -0.046, Net: +2] (Eval took 15.3s)

📊 [EVALUATION @ Step 503,808 | Rollout SPS: 2320 | Tiers: ['heuristic', 'champion']]
   ⚔️  vs Heuristic [FILTER] | WR:  80.0% | Reward: +3.076 | Goals: 117 Scored, 9 Conceded (+108 Net)
   ❌ Retaining current baseline. Did not pass criteria for champion: [WR: None, Reward: None, Net: None] (Eval took 6.7s)

📊 [EVALUATION @ Step 761,856 | Rollout 

In [18]:
import os
import torch
from src.rl_transformer.visualization import evaluate_and_generate_html

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

html_path = evaluate_and_generate_html(
    red_agent="models/stage1/phase3/final_model.pt",
    blue_agent="models/stage1/phase3/pool/history_21762048.pt",
    red_team_size=1,
    blue_team_size=1,
    device=device,
    filename="stage1_selfplay_test_3.html",
    num_episodes=10,
    max_steps=2400,
    action_repeat=4,
    pitch_width=1200.0,
    pitch_height=800.0,
    goal_height=220.0,
)

🎬 Replay generated successfully: /home/minh-quan/Documents/Haxball project/Notebooks/training_transformer/render/stage1_selfplay_test_3.html
